In [27]:
# -*- coding: utf-8 -*-
from typing import Optional, Sequence, List, Tuple
import pandas as pd
from sqlalchemy import create_engine, text
from DATA.stock_invest_function import *

TABLE_NAME = "us_valuation_result"


TABLE_NAME = "us_valuation_result"

# --- DB 연결 유틸 ---
def _engine(db_info: dict):
    url = (
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
        f"@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
    )
    return create_engine(url)

def _latest_created_at(conn, table: str) -> Optional[pd.Timestamp]:
    q = text(f"SELECT MAX(created_at) AS mx FROM {table}")
    s = pd.read_sql(q, conn)['mx'].iloc[0]
    return None if pd.isna(s) else s

# --- 상위 랭킹 메인 함수 ---
def get_top_by_category(
    db_info: dict,
    category: str,                 # 'valuation' 또는 'revenue'
    top_n: int = 20,
    created_at: Optional[str] = None,
    table_name: str = TABLE_NAME,
) -> pd.DataFrame:
    """
    category='valuation'  -> avg_top3 순위(top_n개)
    category='revenue'    -> avg_of_4 순위(top_n개)
    """

    category = category.lower()
    if category not in {"valuation", "revenue"}:
        raise ValueError("category는 'valuation' 또는 'revenue'만 허용됩니다.")

    eng = _engine(db_info)
    with eng.begin() as conn:
        if created_at is None:
            created_at = _latest_created_at(conn, table_name)
            if created_at is None:
                return pd.DataFrame(columns=["ticker", "score"])

        # 1) 우선: 미리 계산된 평균값이 테이블에 있으면 그걸 사용
        target_model = "avg_top3" if category == "valuation" else "avg_of_4"
        q_pre = text(f"""
            SELECT ticker, end_value AS score
              FROM {table_name}
             WHERE category = :cat
               AND model    = :mdl
               AND created_at = :ca
        """)
        pre = pd.read_sql(q_pre, conn, params={"cat": category, "mdl": target_model, "ca": created_at})

        if not pre.empty:
            return pre.sort_values("score", ascending=False).head(top_n).reset_index(drop=True)

        # 2) 없다면: 개별 모델(sarima/lstm/prophet/es)로 계산
        q_raw = text(f"""
            SELECT ticker, model, end_value
              FROM {table_name}
             WHERE category = :cat
               AND model IN ('sarima','lstm','prophet','es')
               AND created_at = :ca
        """)
        df = pd.read_sql(q_raw, conn, params={"cat": category, "ca": created_at})

    if df.empty:
        return pd.DataFrame(columns=["ticker", "score"])

    # 피벗 후 평균 계산
    piv = df.pivot_table(index="ticker", columns="model", values="end_value", aggfunc="last")
    have = [c for c in ["sarima", "lstm", "prophet", "es"] if c in piv.columns]
    if not have:
        return pd.DataFrame(columns=["ticker", "score"])

    if category == "valuation":
        # avg_top3 = 큰 값 3개 평균 (모델이 3개 미만이면 있는 것만 평균)
        def _top3(s: pd.Series) -> float:
            vals = s.dropna().sort_values(ascending=False)
            if len(vals) == 0:
                return float("nan")
            k = min(3, len(vals))
            return float(vals.iloc[:k].mean())
        score = piv[have].apply(_top3, axis=1)
    else:  # category == 'revenue'
        # avg_of_4 = 4개 평균 (결측은 무시; 모델 적으면 있는 것만 평균)
        score = piv[have].mean(axis=1, skipna=True)

    out = (
        score.rename("score")
        .reset_index()
        .sort_values("score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return out


def fetch_all_valuation_results(
    db_info: dict,
    table_name: str = TABLE_NAME,
    columns: Optional[Sequence[str]] = None,   # None이면 *
    category: Optional[str] = None,            # 'valuation' | 'revenue'
    tickers: Optional[Sequence[str]] = None,   # ['AAPL','MSFT'] 등
    created_at: Optional[str] = None,          # 정확 일치 (예: '2025-10-12 15:24:18')
    created_at_like: Optional[str] = None,     # LIKE (예: '2025-10-12%')
    created_ts_like: Optional[str] = None,     # '2025-10-13%' 등
    since: Optional[str] = None,               # start_month_end 하한 (예: '2015-01-01')
    until: Optional[str] = None,               # start_month_end 상한 (예: '2025-12-31')
    order_by: Optional[str] = None,            # 정렬 SQL (None이면 기본 정렬)
    chunksize: int = 100_000,                  # 청크 크기 (메모리 보호)
    to_csv_path: Optional[str] = None,         # 저장 경로 (선택)
    to_excel_path: Optional[str] = None,       # 저장 경로 (선택)
) -> pd.DataFrame:
    """
    us_valuation_result 전체(또는 필터된) 데이터를 DataFrame으로 반환.
    - 대용량 안전: pandas.read_sql(..., chunksize=...)로 청크 결합
    - created_at/created_ts, category, tickers, 날짜 범위 등의 필터 제공
    - 필요 시 CSV/Excel로 바로 저장
    """
    eng = _engine(db_info)

    # SELECT 절
    select_cols = "*"
    if columns:
        select_cols = ", ".join([f"`{c}`" for c in columns])

    # WHERE 절 구성
    where = []
    params = {}

    if category:
        where.append("category = :category")
        params["category"] = category

    if tickers:
        # IN 절 파라미터 바인딩
        tickers = list(tickers)
        tick_params = {f"tk{i}": t for i, t in enumerate(tickers)}
        where.append("ticker IN (" + ", ".join([f":{k}" for k in tick_params.keys()]) + ")")
        params.update(tick_params)

    if created_at:
        where.append("created_at = :created_at")
        params["created_at"] = created_at

    if created_at_like:
        where.append("created_at LIKE :created_at_like")
        params["created_at_like"] = created_at_like

    if created_ts_like:
        where.append("created_ts LIKE :created_ts_like")
        params["created_ts_like"] = created_ts_like

    if since:
        where.append("start_month_end >= :since")
        params["since"] = since

    if until:
        where.append("start_month_end <= :until")
        params["until"] = until

    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    # ORDER BY 기본값
    if order_by is None:
        order_by = "ORDER BY created_at DESC, category, model, ticker, start_month_end"

    sql = f"""
        SELECT {select_cols}
          FROM {table_name}
          {where_sql}
          {order_by}
    """

    # 청크로 읽어 결합
    frames: List[pd.DataFrame] = []
    with eng.begin() as conn:
        for chunk in pd.read_sql(text(sql), conn, params=params, chunksize=chunksize):
            frames.append(chunk)
    df = pd.concat(frames, axis=0, ignore_index=True) if frames else pd.DataFrame()

    # 필요 시 저장
    if to_csv_path:
        df.to_csv(to_csv_path, index=False, encoding="utf-8-sig")
    if to_excel_path:
        df.to_excel(to_excel_path, index=False)

    return df

In [21]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}


In [31]:
# 1) 카테고리별 상위표 (가장 최근 배치)
#
# # valuation에서 avg_top3 상위 30개
# top_val30 = get_top_by_category(db_info, category="valuation", top_n=20)
# print(top_val30)
#
# # revenue에서 avg_of_4 상위 50개 (가장 최근 배치)
# top_rev50 = get_top_by_category(db_info, category="revenue", top_n=50)
# print(top_rev50)

# 특정 created_at 기준으로 보고 싶으면
# top_val = get_top_by_category(db_info, "valuation", 20, created_at="2025-10-12 15:24:18")

   ticker     score
0    JBTM  7.168817
1    IDCC  6.761535
2    ITRI  5.154611
3    IPAR  4.679056
4    INSP  4.526233
5    ITGR  4.455977
6     KAI  4.335320
7    INDB  3.788943
8    IOSP  3.275599
9     JOE  2.923439
10   JJSF  2.891630
11   INSW  2.807066
12   JBLU  2.141875
13   JBSS  0.983241
14   IIIN  0.715885
15   INVA  0.154402
   ticker      score
0    JBLU  10.259207
1    JBTM   2.956352
2    ITRI   2.329004
3    IOSP   1.999204
4    ITGR   1.931974
5    JJSF   1.743063
6    IPAR   1.512928
7    INDB   1.107163
8    JBSS   1.103840
9     KAI   1.101305
10   INSP   1.063438
11   INSW   0.924898
12   IDCC   0.790628
13   IIIN   0.652676
14   INVA   0.380584
15    JOE   0.309077


In [35]:
# 1) 진짜 전체 다 보기 (기본 정렬, 메모리 안전)
all_df = fetch_all_valuation_results(db_info)


In [55]:
rev_tops = all_df[all_df['model'] == 'avg_of_4'].sort_values(by='growth', ascending=False)all_df['model'] == 'avg_top3'].sort_values(by='growth', ascending=False)

In [60]:
rev_tops

,id,ticker,category,model,start_month_end,start_value,end_value,growth,created_at,created_ts,updated_ts
1191,6850,VIR,revenue,avg_of_4,2025-10-31,0.068961,0.161716,1.345023,2025-10-12 12:35:39,2025-10-12 21:35:39,2025-10-12 21:35:39
6604,1570,MRNA,revenue,avg_of_4,2025-10-31,2.838970,4.749586,0.672996,2025-10-12 00:41:49,2025-10-12 09:41:49,2025-10-12 09:41:49
2791,5250,RDN,revenue,avg_of_4,2025-10-31,3.832665,6.150602,0.604785,2025-10-12 10:26:29,2025-10-12 19:26:28,2025-10-12 19:26:28
1209,7030,WOR,revenue,avg_of_4,2025-10-31,1.451050,2.138648,0.473862,2025-10-12 12:35:39,2025-10-12 21:35:39,2025-10-12 21:35:39
2397,5710,SEDG,revenue,avg_of_4,2025-10-31,1.380570,1.786608,0.294109,2025-10-12 10:55:44,2025-10-12 19:55:43,2025-10-12 19:55:43
...,...,...,...,...,...,...,...,...,...,...,...
4806,3400,MDU,revenue,avg_of_4,2025-10-31,1.370505,0.796666,-0.418706,2025-10-12 07:57:56,2025-10-12 16:57:56,2025-10-12 16:57:56
4209,4030,WRK,revenue,avg_of_4,2025-10-31,NaN,NaN,NaN,2025-10-12 08:31:40,2025-10-12 17:31:39,2025-10-12 17:31:39
4592,3460,CDAY,revenue,avg_of_4,2025-10-31,NaN,NaN,NaN,2025-10-12 08:01:03,2025-10-12 17:01:03,2025-10-12 17:01:03
4604,3580,PEAK,revenue,avg_of_4,2025-10-31,NaN,NaN,NaN,2025-10-12 08:01:03,2025-10-12 17:01:03,2025-10-12 17:01:03


In [64]:
tckr = 'ANET'

value_tops[value_tops['ticker'] == tckr]

,id,ticker,category,model,start_month_end,start_value,end_value,growth,created_at,created_ts,updated_ts
6891,1235,ANET,valuation,avg_top3,2025-10-31,199.131837,259.246781,0.301885,2025-10-12 00:21:23,2025-10-12 09:21:23,2025-10-12 09:21:23
